# VocalCoach Colab Training

Two-stage training to solve the pitch/technique multi-task conflict:
- **Stage 1**: Train pitch+VAD only (no technique) for 50 epochs so the backbone establishes clean pitch representations
- **Stage 2**: Resume from stage 1 checkpoint and add the technique head for 50 more epochs

Run cells in order. Cells 1–4 are setup (run once per session). Then pick **one** of the experiment sections.

**Before starting**: upload `NanoPitch_data.zip` to your Google Drive root.
Zip created locally with:
```bash
cd ~/NanoPitch-MusicalAI
zip -j -1 -v NanoPitch_data.zip \
    data/clean.npz data/noise.npz data/test.npz \
    data/vocalset/technique_train.npz \
    data/vocalset/technique_test.npz
```

## Cell 1 — GPU check

In [1]:
!nvidia-smi
import torch
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU:            {torch.cuda.get_device_name(0)}")
print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tue May 12 04:08:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!ls -lh /content/data/

ls: cannot access '/content/data/': No such file or directory


## Cell 2 — Mount Drive and extract data

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

os.makedirs('/content/data/vocalset', exist_ok=True)

print("Extracting NanoPitch_data.zip...")
with zipfile.ZipFile('/content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch_data.zip', 'r') as z:
    for name in ['clean.npz', 'noise.npz', 'test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/')
    for name in ['technique_train.npz', 'technique_test.npz']:
        print(f"  {name}...")
        z.extract(name, '/content/data/vocalset/')

print("\nExtraction complete.")
!ls -lh /content/data/
!ls -lh /content/data/vocalset/

Mounted at /content/drive
Extracting NanoPitch_data.zip...
  clean.npz...
  noise.npz...
  test.npz...
  technique_train.npz...
  technique_test.npz...

Extraction complete.
total 3.4G
-rw-r--r-- 1 root root 2.5G May 12 04:10 clean.npz
-rw-r--r-- 1 root root 960M May 12 04:10 noise.npz
-rw-r--r-- 1 root root  41M May 12 04:10 test.npz
drwxr-xr-x 2 root root 4.0K May 12 04:10 vocalset
total 90M
-rw-r--r-- 1 root root 17M May 12 04:10 technique_test.npz
-rw-r--r-- 1 root root 73M May 12 04:10 technique_train.npz


## Cell 3 — Clone repo and install dependencies

In [4]:
!mkdir -p /content/NanoPitch-MusicalAI
%cd /content/NanoPitch-MusicalAI

/content/NanoPitch-MusicalAI


In [5]:
!git clone https://github.com/rajat17-personal/NanoPitch-MusicalAI -b feat/finalProject /content/NanoPitch-MusicalAI
%cd /content/NanoPitch-MusicalAI
!pip install -r requirements.txt --quiet
print("Setup complete.")

Cloning into '/content/NanoPitch-MusicalAI'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 74 (delta 11), reused 30 (delta 8), pack-reused 32 (from 2)
Receiving objects: 100% (74/74), 5.03 MiB | 5.09 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/NanoPitch-MusicalAI
Setup complete.


## Cell 4 — Verify data loads correctly

In [6]:
import numpy as np

clean = np.load('/content/data/clean.npz')
print(f"clean.npz keys:    {list(clean.keys())}")
print(f"  mel shape:       {clean['mel'].shape}")
print(f"  lengths shape:   {clean['lengths'].shape}")

tech = np.load('/content/data/vocalset/technique_train.npz')
print(f"\ntechnique_train.npz keys: {list(tech.keys())}")
print(f"  clips:           {tech['lengths'].shape[0]}")

test = np.load('/content/data/test.npz')
print(f"\ntest.npz keys:     {list(test.keys())}")
print(f"  clips:           {test['clips'].shape[0]}")

clean.npz keys:    ['mel', 'f0', 'vad', 'lengths']
  mel shape:       (36372498, 40)
  lengths shape:   (40940,)

technique_train.npz keys: ['mel', 'f0', 'vad', 'technique', 'lengths', 'singers']
  clips:           824

test.npz keys:     ['clips', 'clean', 'f0', 'vad', 'snr', 'snr_levels', 'clip_len']
  clips:           600


---
## Experiment: Option A — Two-stage training

**Why two stages?** Joint training from epoch 1 causes the technique gradient (~10×
stronger than pitch at epoch 1) to immediately capture the backbone, collapsing
pitch posteriors. Stage 1 lets pitch establish clean representations first.

Run the architecture you want (TCN or Conformer), or both in parallel if you have two sessions.

Checkpoints write directly to Drive so a session disconnect loses at most one epoch.

In [7]:
!ls -all /content/NanoPitch-MusicalAI

total 132
drwxr-xr-x 8 root root  4096 May 12 04:10 .
drwxr-xr-x 1 root root  4096 May 12 04:10 ..
drwxr-xr-x 4 root root  4096 May 12 04:10 deployment
drwxr-xr-x 8 root root  4096 May 12 04:10 .git
-rw-r--r-- 1 root root    93 May 12 04:10 .gitignore
-rw-r--r-- 1 root root 19126 May 12 04:10 LICENSE
drwxr-xr-x 2 root root  4096 May 12 04:10 notebooks
-rw-r--r-- 1 root root 23128 May 12 04:10 README.md
-rw-r--r-- 1 root root   162 May 12 04:10 requirements.txt
-rw-r--r-- 1 root root 21186 May 12 04:10 RESULTS.MD
drwxr-xr-x 2 root root  4096 May 12 04:10 scripts
drwxr-xr-x 2 root root  4096 May 12 04:10 training
drwxr-xr-x 2 root root  4096 May 12 04:10 vocalcoach
-rw-r--r-- 1 root root 22620 May 12 04:10 VOCALCOACH_PLAN.md


### TCN — Stage 1: pitch + VAD only (50 epochs)

In [8]:
!python /content/NanoPitch-MusicalAI/vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/tcn_stage1_pitchonly \
    --epochs 50 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

2026-05-12 04:11:11.048307: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 04:11:11.120757: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Device: cuda  |  Arch: tcn  |  causal=False
VocalCoachTCN: 449,006 parameters (hidden=128, blocks=8, causal=False)
NoisePool: 15,283,949 frames (42.5h), 12214 usable segments, RAM=1166 MB
Augmentation: noise_specaug  SNR=[-10.0,30.0] dB  p_clean=0.0
PitchVADDataset: 36,372,498 frames, 39511 usable segments, RAM=2914 MB
Pitch/VAD-only dataset: 20000 samples — technique hea

### TCN — Stage 2: resume + technique head (epochs 51–100)

Run this after Stage 1 completes. Can be a new session — data and repo setup (cells 1–3) must be re-run, but the checkpoint is already on Drive.

In [ ]:
!python vocalcoach/train.py \
    --arch tcn --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/NanoPitch-runs/tcn_stage2_technique \
    --epochs 100 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --patience 0 \
    --resume /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/tcn_stage1_pitchonly/checkpoints/best_loss.pth

2026-05-12 04:31:23.225058: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 04:31:23.300123: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Device: cuda  |  Arch: tcn  |  causal=False
VocalCoachTCN: 449,006 parameters (hidden=128, blocks=8, causal=False)
/content/NanoPitch-MusicalAI/vocalcoach/train.py:815: RuntimeWarning: Loading checkpoint executes Python deserialization — only use checkpoints from trusted sources.
  warnings.warn(
Resumed from epoch 49
NoisePool: 15,283,949 frames (42.5h), 12214 usable seg

### Conformer — Stage 1: pitch + VAD only (50 epochs)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage1_pitchonly \
    --epochs 50 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 \
    --augment noise_specaug \
    --patience 0

### Conformer — Stage 2: resume + technique head (epochs 51–100)

In [ ]:
!python vocalcoach/train.py \
    --arch conformer --causal false \
    --data-dir /content/data \
    --technique-dirs /content/data/vocalset \
    --output-dir /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage2_technique \
    --epochs 100 --batch-size 32 \
    --w-pitch 2 --w-vad 0.05 --w-technique 2 \
    --augment noise_specaug \
    --technique-pos-weights 2.9 4.2 1.0 4.0 1.9 \
    --patience 0 \
    --resume /content/drive/MyDrive/musicalAI/vocalCoach/NanoPitch-runs/conformer_stage1_pitchonly/checkpoints/best_loss.pth

---
## Evaluate a completed run

Copy the run directory from Drive to local disk first (faster I/O than reading from Drive directly).

In [ ]:
RUN_NAME = "tcn_stage2_technique"  # change to the run you want to evaluate

import shutil
shutil.copytree(
    f'/content/drive/MyDrive/NanoPitch-runs/{RUN_NAME}',
    f'/content/runs/{RUN_NAME}',
    dirs_exist_ok=True
)

!python vocalcoach/evaluate.py \
    --checkpoint /content/runs/{RUN_NAME}/checkpoints/best_metric.pth \
    --data-dir /content/data \
    --technique-dir /content/data/vocalset